In [1]:
# ==========================================================
# BLOQUE 1. Configuración
# Admisión Máster
# ==========================================================

import json
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin


# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

URL = (
    "https://www.upv.es/admision/admision-master/index-es.html"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0"
    )
}

from google.colab import drive

drive.mount("/content/drive")

# ----------------------------------------------------------
# Funciones auxiliares
# ----------------------------------------------------------

def texto_limpio(elemento):

    if elemento is None:
        return ""

    return " ".join(
        elemento.stripped_strings
    )


def url_absoluta(url):

    if not url:
        return ""

    return urljoin(
        URL,
        url
    )


def primer_enlace(elemento):

    if elemento is None:
        return ""

    enlace = elemento.find(
        "a",
        href=True
    )

    if enlace:
        return url_absoluta(
            enlace["href"]
        )

    return ""


def obtener_descripcion(contenedor):

    if contenedor is None:
        return ""

    parrafos = contenedor.find_all(
        "p",
        recursive=False
    )

    texto = " ".join(
        texto_limpio(p)
        for p in parrafos
    )

    return texto.strip()

Mounted at /content/drive


In [2]:
# ==========================================================
# BLOQUE 2. Ruta del JSON
# ==========================================================

import os


NOMBRE_PROGRAMA = "Extrae_Admision_Master.ipynb"


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root
        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(
    ruta_programa,
    "JSONs"
)


os.makedirs(
    CARPETA_JSON,
    exist_ok=True
)


RUTA_JSON = os.path.join(
    CARPETA_JSON,
    "admision_master.json"
)


print("Directorio del proyecto:")
print(ruta_programa)

print()

print("JSON:")
print(RUTA_JSON)

Directorio del proyecto:
/content/drive/MyDrive/TFG Teleco

JSON:
/content/drive/MyDrive/TFG Teleco/JSONs/admision_master.json


In [3]:
# ==========================================================
# BLOQUE 4. Descarga y localización de secciones
# ==========================================================

respuesta = requests.get(
    URL,
    headers=HEADERS
)

respuesta.raise_for_status()

soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)


# ----------------------------------------------------------
# Padre
# ----------------------------------------------------------

titulo_padre = texto_limpio(
    soup.find("h1")
)

descripcion_padre = texto_limpio(
    soup.find("h2")
)


# ----------------------------------------------------------
# Secciones
# ----------------------------------------------------------

secciones = []

for i in range(1, 7):

    identificador = f"section-{i:02d}"

    seccion = soup.find(
        "section",
        id=identificador
    )

    if seccion is None:

        print(
            f"No encontrada {identificador}"
        )

        continue

    secciones.append(
        seccion
    )


# ----------------------------------------------------------
# Validación
# ----------------------------------------------------------

print("Título:", titulo_padre)
print("Descripción:", descripcion_padre)
print()

print(
    "Secciones encontradas:",
    len(secciones)
)

for s in secciones:

    titulo = s.find(
        ["h2", "h3"]
    )

    print(
        "-",
        s["id"],
        "|",
        texto_limpio(titulo)
    )

Título: Admisión a máster
Descripción: Continúa tus estudios. Conviértete en especialista.

Secciones encontradas: 6
- section-01 | Conoce la UPV
- section-02 | Elige estudios
- section-03 | Haz tu preinscripción
- section-04 | Realiza la matrícula
- section-05 | Consulta las becas
- section-06 | Comienza tu posgrado


In [4]:
# ==========================================================
# BLOQUE 5. Funciones de extracción
# ==========================================================

def extraer_tarjetas(seccion):

    tarjetas = []

    for caja in seccion.select(".box-number-box"):

        titulo = texto_limpio(
            caja.find("h3")
        )

        if not titulo:
            continue

        descripcion = obtener_descripcion(
            caja
        )

        url = primer_enlace(
            caja
        )

        tarjetas.append({
            "titulo": titulo,
            "descripcion": descripcion,
            "url": url
        })

    return tarjetas


def extraer_banners(seccion):

    banners = []

    for banner in seccion.select(".banner--content"):

        titulo = texto_limpio(
            banner.find("h3")
        )

        descripcion = obtener_descripcion(
            banner
        )

        url = primer_enlace(
            banner
        )

        banners.append({
            "titulo": titulo,
            "descripcion": descripcion,
            "url": url
        })

    return banners


def extraer_acordeones(seccion):

    acordeones = []

    for bloque in seccion.select(".accordion-element-content"):

        titulo = texto_limpio(
            bloque.find("h3")
        )

        texto = texto_limpio(
            bloque
        )

        enlaces = []

        for a in bloque.find_all(
            "a",
            href=True
        ):

            enlaces.append({

                "texto": texto_limpio(a),

                "url": url_absoluta(
                    a["href"]
                )

            })

        acordeones.append({

            "titulo": titulo,

            "texto": texto,

            "enlaces": enlaces

        })

    return acordeones


def extraer_enlaces(seccion):

    enlaces = []

    vistos = set()

    for a in seccion.find_all(
        "a",
        href=True
    ):

        texto = texto_limpio(a)

        url = url_absoluta(
            a["href"]
        )

        if not texto or url in vistos:

            continue

        vistos.add(url)

        enlaces.append({

            "texto": texto,

            "url": url

        })

    return enlaces

In [5]:
# ==========================================================
# BLOQUE 6. Construcción del JSON
# ==========================================================

padre = {

    "titulo": titulo_padre,

    "url": URL,

    "descripcion": descripcion_padre,

    "secciones": []

}


for seccion in secciones:


    titulo = texto_limpio(
        seccion.find(
            ["h2", "h3"]
        )
    )


    descripcion = obtener_descripcion(
        seccion
    )


    datos_seccion = {

        "id": seccion.get("id"),

        "titulo": titulo,

        "descripcion": descripcion,

        "tarjetas": extraer_tarjetas(
            seccion
        ),

        "acordeones": extraer_acordeones(
            seccion
        ),

        "banners": extraer_banners(
            seccion
        ),

        "enlaces": extraer_enlaces(
            seccion
        )

    }


    padre["secciones"].append(
        datos_seccion
    )


datos = {

    "padres": [
        padre
    ]

}


print()

print(
    "Secciones:",
    len(
        padre["secciones"]
    )
)

for s in padre["secciones"]:

    print()

    print(
        s["titulo"]
    )

    print(
        "Tarjetas:",
        len(
            s["tarjetas"]
        )
    )

    print(
        "Acordeones:",
        len(
            s["acordeones"]
        )
    )

    print(
        "Banners:",
        len(
            s["banners"]
        )
    )

    print(
        "Enlaces:",
        len(
            s["enlaces"]
        )
    )


Secciones: 6

Conoce la UPV
Tarjetas: 1
Acordeones: 0
Banners: 0
Enlaces: 7

Elige estudios
Tarjetas: 1
Acordeones: 0
Banners: 0
Enlaces: 10

Haz tu preinscripción
Tarjetas: 1
Acordeones: 5
Banners: 1
Enlaces: 19

Realiza la matrícula
Tarjetas: 1
Acordeones: 2
Banners: 1
Enlaces: 8

Consulta las becas
Tarjetas: 1
Acordeones: 0
Banners: 0
Enlaces: 7

Comienza tu posgrado
Tarjetas: 1
Acordeones: 0
Banners: 0
Enlaces: 5


In [6]:
# ==========================================================
# BLOQUE 7. Guardado JSON
# ==========================================================

with open(

    RUTA_JSON,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        datos,

        f,

        ensure_ascii=False,

        indent=4

    )


print()

print(
    "JSON guardado en:"
)

print(
    RUTA_JSON
)


JSON guardado en:
/content/drive/MyDrive/TFG Teleco/JSONs/admision_master.json


In [7]:
# ==========================================================
# BLOQUE 8. Validación del JSON
# ==========================================================

print("=" * 70)
print("VALIDACIÓN JSON ADMISIÓN MÁSTER")
print("=" * 70)

for padre in datos["padres"]:

    print()
    print("PADRE")
    print("-" * 70)

    print("Título:", padre["titulo"])
    print("URL:", padre["url"])

    print()

    for seccion in padre["secciones"]:

        print(seccion["titulo"])

        print(
            "  descripción:",
            len(seccion["descripcion"])
        )

        print(
            "  tarjetas:",
            len(seccion["tarjetas"])
        )

        print(
            "  acordeones:",
            len(seccion["acordeones"])
        )

        print(
            "  banners:",
            len(seccion["banners"])
        )

        print(
            "  enlaces:",
            len(seccion["enlaces"])
        )

        print()

VALIDACIÓN JSON ADMISIÓN MÁSTER

PADRE
----------------------------------------------------------------------
Título: Admisión a máster
URL: https://www.upv.es/admision/admision-master/index-es.html

Conoce la UPV
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 0
  enlaces: 7

Elige estudios
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 0
  enlaces: 10

Haz tu preinscripción
  descripción: 0
  tarjetas: 1
  acordeones: 5
  banners: 1
  enlaces: 19

Realiza la matrícula
  descripción: 0
  tarjetas: 1
  acordeones: 2
  banners: 1
  enlaces: 8

Consulta las becas
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 0
  enlaces: 7

Comienza tu posgrado
  descripción: 0
  tarjetas: 1
  acordeones: 0
  banners: 0
  enlaces: 5



In [8]:
# ==========================================================
# BLOQUE 9. Generación Markdown semántico para RAG
# Admisión Máster + metadatos YAML
# ==========================================================

import json
import os
import re


# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

ruta_json = RUTA_JSON

directorio_base = os.path.join(
    ruta_programa,
    "ADMISION",
    "Master"
)

os.makedirs(
    directorio_base,
    exist_ok=True
)


# ----------------------------------------------------------
# Parámetros
# ----------------------------------------------------------

CATEGORIA = "admision"
NIVEL = "master"

INTRO_DOCUMENTO = (
    "Información completa sobre el proceso de admisión "
    "a estudios oficiales de máster en la "
    "Universitat Politècnica de València."
)


# ----------------------------------------------------------
# Limpieza nombres
# ----------------------------------------------------------

def limpiar_nombre(nombre):

    nombre = nombre.lower()

    cambios = {
        "á":"a",
        "é":"e",
        "í":"i",
        "ó":"o",
        "ú":"u",
        "ñ":"n"
    }

    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )

    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )

    return nombre.strip("_")


# ----------------------------------------------------------
# Metadatos YAML
# ----------------------------------------------------------

def escribir_metadatos(
    f,
    tipo_documento,
    seccion=None
):

    f.write("---\n")

    f.write("fuente: UPV\n")

    f.write(
        f"categoria: {CATEGORIA}\n"
    )

    f.write(
        f"nivel: {NIVEL}\n"
    )

    f.write(
        f"tipo_documento: {tipo_documento}\n"
    )

    if seccion:

        f.write(
            f"seccion: {seccion}\n"
        )

    f.write("---\n\n")


# ----------------------------------------------------------
# Cargar JSON
# ----------------------------------------------------------

with open(
    ruta_json,
    encoding="utf-8"
) as f:

    datos = json.load(f)


contador = 0


# ==========================================================
# Generación
# ==========================================================

for padre in datos["padres"]:

    nombre_padre = limpiar_nombre(
        padre["titulo"]
    )


    # ------------------------------------------------------
    # Documento padre
    # ------------------------------------------------------

    archivo_padre = os.path.join(

        directorio_base,

        f"{nombre_padre}.md"

    )


    with open(
        archivo_padre,
        "w",
        encoding="utf-8"
    ) as f:


        escribir_metadatos(
            f,
            "padre"
        )


        f.write(
            f"# {padre['titulo']}\n\n"
        )


        f.write(
            INTRO_DOCUMENTO
            +
            "\n\n"
        )


        for seccion in padre["secciones"]:


            f.write(
                f"## {seccion['titulo']}\n\n"
            )


            if seccion.get(
                "descripcion"
            ):

                f.write(
                    seccion["descripcion"]
                    +
                    "\n\n"
                )


            # Tarjetas

            for t in seccion.get(
                "tarjetas",
                []
            ):

                f.write(
                    f"### {t['titulo']}\n\n"
                )


                if t.get(
                    "descripcion"
                ):

                    f.write(
                        t["descripcion"]
                        +
                        "\n\n"
                    )


                if t.get(
                    "url"
                ):

                    f.write(
                        f"Más información: {t['url']}\n\n"
                    )


            # Acordeones

            for acc in seccion.get(
                "acordeones",
                []
            ):


                if acc.get(
                    "titulo"
                ):

                    f.write(
                        f"### {acc['titulo']}\n\n"
                    )


                if acc.get(
                    "texto"
                ):

                    f.write(
                        acc["texto"]
                        +
                        "\n\n"
                    )


                for enlace in acc.get(
                    "enlaces",
                    []
                ):

                    f.write(
                        f"- {enlace['texto']}: "
                        f"{enlace['url']}\n"
                    )

                f.write("\n")


            # Banners

            for b in seccion.get(
                "banners",
                []
            ):


                if b.get(
                    "titulo"
                ):

                    f.write(
                        f"### {b['titulo']}\n\n"
                    )


                if b.get(
                    "descripcion"
                ):

                    f.write(
                        b["descripcion"]
                        +
                        "\n\n"
                    )


                if b.get(
                    "url"
                ):

                    f.write(
                        f"Más información: {b['url']}\n\n"
                    )


    contador += 1


    # ------------------------------------------------------
    # Documentos por sección
    # ------------------------------------------------------

    for seccion in padre["secciones"]:


        nombre_seccion = limpiar_nombre(
            seccion["titulo"]
        )


        archivo = os.path.join(
            directorio_base,
            f"{nombre_seccion}.md"
        )


        with open(
            archivo,
            "w",
            encoding="utf-8"
        ) as f:


            escribir_metadatos(
                f,
                "seccion",
                nombre_seccion
            )


            f.write(
                f"# {seccion['titulo']}\n\n"
            )


            f.write(
                f"Proceso: {padre['titulo']}\n\n"
            )


            if seccion.get(
                "descripcion"
            ):

                f.write(
                    seccion["descripcion"]
                    +
                    "\n\n"
                )


            # Tarjetas

            for t in seccion.get(
                "tarjetas",
                []
            ):

                f.write(
                    f"## {t['titulo']}\n\n"
                )


                if t.get(
                    "descripcion"
                ):

                    f.write(
                        t["descripcion"]
                        +
                        "\n\n"
                    )


                if t.get(
                    "url"
                ):

                    f.write(
                        f"Enlace oficial: {t['url']}\n\n"
                    )


            # Acordeones

            for acc in seccion.get(
                "acordeones",
                []
            ):


                if acc.get(
                    "titulo"
                ):

                    f.write(
                        f"## {acc['titulo']}\n\n"
                    )


                if acc.get(
                    "texto"
                ):

                    f.write(
                        acc["texto"]
                        +
                        "\n\n"
                    )


                for enlace in acc.get(
                    "enlaces",
                    []
                ):

                    f.write(
                        f"- {enlace['texto']}: {enlace['url']}\n"
                    )

                f.write("\n")


            # Banners

            for b in seccion.get(
                "banners",
                []
            ):


                if b.get(
                    "titulo"
                ):

                    f.write(
                        f"## {b['titulo']}\n\n"
                    )


                if b.get(
                    "descripcion"
                ):

                    f.write(
                        b["descripcion"]
                        +
                        "\n\n"
                    )


                if b.get(
                    "url"
                ):

                    f.write(
                        f"Enlace oficial: {b['url']}\n\n"
                    )


        contador += 1


print()

print(
    "Markdown generado correctamente."
)

print(
    "Archivos creados:",
    contador
)

print(
    "Ruta:",
    directorio_base
)


Markdown generado correctamente.
Archivos creados: 7
Ruta: /content/drive/MyDrive/TFG Teleco/ADMISION/Master


In [9]:
# ==========================================================
# BLOQUE 10. Descarga, filtrado y extracción de recursos
#
# Admisión Máster
# ==========================================================

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse

# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

paginas_extraidas = []

DOMINIOS_UPV = {
    "www.upv.es",
    "upv.es",
    "www.jpa.upv.es",
    "jpa.upv.es"
}

# ----------------------------------------------------------
# Funciones auxiliares
# ----------------------------------------------------------

def normalizar_url_final(url):

    if not url:
        return None

    try:
        p = urlparse(url)

    except Exception:
        return None

    if not p.scheme or not p.netloc:
        return None

    return urlunparse(
        (
            p.scheme.lower(),
            p.netloc.lower(),
            p.path.rstrip("/") or "/",
            "",
            p.query,
            ""
        )
    )


# ----------------------------------------------------------

def es_url_upv(url):

    if not url:
        return False

    try:
        dominio = urlparse(url).netloc.lower()

    except Exception:
        return False

    return dominio in DOMINIOS_UPV


# ----------------------------------------------------------

def eliminar_basura(soup):

    for elemento in soup.find_all(
        [
            "script",
            "style",
            "noscript",
            "header",
            "footer",
            "nav",
            "aside",
            "form",
            "iframe"
        ]
    ):

        elemento.decompose()


# ----------------------------------------------------------

def encontrar_contenido_principal(soup):

    if soup is None:
        return None

    selectores = [
        "main",
        "article",
        "#content",
        ".content",
        ".main-content",
        ".container"
    ]

    for selector in selectores:

        elemento = soup.select_one(
            selector
        )

        if elemento is not None:

            texto = elemento.get_text(
                " ",
                strip=True
            )

            if len(texto) >= 100:

                return elemento

    return None


# ----------------------------------------------------------

def normalizar_texto_para_comparacion(texto):

    if not texto:
        return ""

    return " ".join(
        texto.lower().split()
    )


# ----------------------------------------------------------

def contenido_demasiado_corto(contenido):

    return len(
        contenido.strip()
    ) < 100


# ----------------------------------------------------------

def contenido_claramente_ajeno(
    titulo,
    contenido,
    url
):

    texto = (
        titulo
        + " "
        + contenido
        + " "
        + url
    ).lower()

    patrones_ajenos = [

        # Doctorado / investigación
        "doctorado",
        "tesis doctoral",
        "doctorando",

        # Personal universitario
        "personal docente e investigador",
        "personal investigador",
        "pdi",
        "profesorado",

        # Investigación
        "proyecto de investigación",
        "proyectos de investigación",
        "grupo de investigación",
        "grupos de investigación",

        # Empresas / transferencia
        "empresa de base tecnológica",
        "spin-off",
        "transferencia tecnológica",

        # Alumni
        "antiguos alumnos",
        "exalumnos"
    ]

    coincidencias = sum(
        1
        for patron in patrones_ajenos
        if patron in texto
    )

    # Filtro conservador:
    # una coincidencia aislada no es suficiente.

    if coincidencias >= 2:

        return True

    return False


# ----------------------------------------------------------
# Descargar y validar
# ----------------------------------------------------------

def descargar_y_validar(url):

    try:

        respuesta = requests.get(

            url,

            headers=HEADERS,

            timeout=20,

            allow_redirects=True

        )

        respuesta.raise_for_status()

    except requests.RequestException as e:

        print(
            "Error de descarga:",
            e
        )

        return (
            None,
            None,
            "error de descarga"
        )


    # ------------------------------------------------------
    # URL final después de redirecciones
    # ------------------------------------------------------

    url_final = normalizar_url_final(
        respuesta.url
    )


    # ------------------------------------------------------
    # Comprobar dominio de la URL final
    # ------------------------------------------------------

    if not es_url_upv(
        url_final
    ):

        return (
            url_final,
            None,
            "URL fuera del ámbito útil"
        )


    # ------------------------------------------------------
    # Solo páginas HTML
    # ------------------------------------------------------

    content_type = respuesta.headers.get(
        "Content-Type",
        ""
    ).lower()


    if "text/html" not in content_type:

        return (
            url_final,
            None,
            "contenido no HTML"
        )


    # ------------------------------------------------------
    # Parsear HTML
    # ------------------------------------------------------

    soup = BeautifulSoup(

        respuesta.text,

        "html.parser"

    )


    eliminar_basura(
        soup
    )


    # ------------------------------------------------------
    # Buscar contenido principal
    # ------------------------------------------------------

    contenido_principal = encontrar_contenido_principal(
        soup
    )


    if contenido_principal is None:

        return (
            url_final,
            None,
            "no se encontró contenido principal"
        )


    return (
        url_final,
        contenido_principal,
        None
    )


# ==========================================================
# RECOPILACIÓN DE URLS DEL JSON
# ==========================================================

urls_candidatas = []

urls_vistas = set()


for padre in datos["padres"]:

    for seccion in padre.get(
        "secciones",
        []
    ):

        # --------------------------------------------------
        # URLs de las tarjetas
        # --------------------------------------------------

        for tarjeta in seccion.get(
            "tarjetas",
            []
        ):

            url = tarjeta.get(
                "url",
                ""
            )

            if url:

                url = normalizar_url_final(
                    url
                )

                if (
                    url
                    and url not in urls_vistas
                ):

                    urls_vistas.add(
                        url
                    )

                    urls_candidatas.append({

                        "url": url,

                        "texto": tarjeta.get(
                            "titulo",
                            ""
                        ),

                        "seccion_origen":
                            seccion.get(
                                "titulo",
                                ""
                            ),

                        "padre_origen":
                            padre.get(
                                "titulo",
                                ""
                            ),

                        "tipo_origen":
                            "tarjeta"

                    })


        # --------------------------------------------------
        # URLs de los acordeones
        # --------------------------------------------------

        for acordeon in seccion.get(
            "acordeones",
            []
        ):

            for enlace in acordeon.get(
                "enlaces",
                []
            ):

                url = enlace.get(
                    "url",
                    ""
                )

                if not url:
                    continue

                url = normalizar_url_final(
                    url
                )

                if (
                    not url
                    or url in urls_vistas
                ):

                    continue

                urls_vistas.add(
                    url
                )

                urls_candidatas.append({

                    "url": url,

                    "texto": enlace.get(
                        "texto",
                        ""
                    ),

                    "seccion_origen":
                        seccion.get(
                            "titulo",
                            ""
                        ),

                    "padre_origen":
                        padre.get(
                            "titulo",
                            ""
                        ),

                    "tipo_origen":
                        "acordeon"

                })


        # --------------------------------------------------
        # URLs de banners
        # --------------------------------------------------

        for banner in seccion.get(
            "banners",
            []
        ):

            url = banner.get(
                "url",
                ""
            )

            if not url:
                continue

            url = normalizar_url_final(
                url
            )

            if (
                not url
                or url in urls_vistas
            ):

                continue

            urls_vistas.add(
                url
            )

            urls_candidatas.append({

                "url": url,

                "texto": banner.get(
                    "titulo",
                    ""
                ),

                "seccion_origen":
                    seccion.get(
                        "titulo",
                        ""
                    ),

                "padre_origen":
                    padre.get(
                        "titulo",
                        ""
                    ),

                "tipo_origen":
                    "banner"

            })


        # --------------------------------------------------
        # Enlaces generales de la sección
        # --------------------------------------------------

        for enlace in seccion.get(
            "enlaces",
            []
        ):

            url = enlace.get(
                "url",
                ""
            )

            if not url:
                continue

            url = normalizar_url_final(
                url
            )

            if (
                not url
                or url in urls_vistas
            ):

                continue

            urls_vistas.add(
                url
            )

            urls_candidatas.append({

                "url": url,

                "texto": enlace.get(
                    "texto",
                    ""
                ),

                "seccion_origen":
                    seccion.get(
                        "titulo",
                        ""
                    ),

                "padre_origen":
                    padre.get(
                        "titulo",
                        ""
                    ),

                "tipo_origen":
                    "enlace"

            })


print()
print("=" * 80)
print("RECOPILACIÓN DE URLS")
print("=" * 80)

print(
    "URLs candidatas:",
    len(urls_candidatas)
)


# ==========================================================
# DESCARGA Y FILTRADO
# ==========================================================

urls_finales_procesadas = set()


for enlace in urls_candidatas:

    url_original = enlace["url"]


    print()
    print("=" * 80)
    print(url_original)


    # ------------------------------------------------------
    # Comprobar ámbito inicial
    # ------------------------------------------------------

    if not es_url_upv(
        url_original
    ):

        print(
            "Descartado (URL fuera del ámbito útil)"
        )

        continue


    # ------------------------------------------------------
    # Descargar y validar
    # ------------------------------------------------------

    url_final, contenido_principal, motivo = (
        descargar_y_validar(
            url_original
        )
    )


    if motivo:

        print(
            f"Descartado ({motivo})"
        )

        continue


    print(
        "URL final:",
        url_final
    )


    # ------------------------------------------------------
    # Evitar duplicados después de redirecciones
    # ------------------------------------------------------

    if url_final in urls_finales_procesadas:

        print(
            "Descartado (URL final duplicada)"
        )

        continue


    urls_finales_procesadas.add(
        url_final
    )


    # ------------------------------------------------------
    # Título
    # ------------------------------------------------------

    titulo = ""


    h1 = contenido_principal.find(
        "h1"
    )


    if h1:

        titulo = limpiar_nombre(
            h1.get_text(" ")
        )


    # Si no se encuentra h1,
    # utilizar el título del enlace.

    if not titulo:

        titulo = enlace.get(
            "texto",
            ""
        )


    # Recuperar texto limpio del título

    if h1:

        titulo = " ".join(
            h1.stripped_strings
        )

    else:

        titulo = " ".join(
            titulo.split()
        )


    # ------------------------------------------------------
    # Extracción estructurada
    # ------------------------------------------------------

    bloques = []


    for elemento in contenido_principal.find_all(

        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "li",
            "table"
        ]

    ):

        texto = " ".join(
            elemento.stripped_strings
        )


        if len(texto) < 3:

            continue


        # Evitar repetir el H1 como contenido

        if (

            elemento.name == "h1"

            and normalizar_texto_para_comparacion(
                texto
            )
            ==
            normalizar_texto_para_comparacion(
                titulo
            )

        ):

            continue


        bloques.append(
            texto
        )


    # ------------------------------------------------------
    # Eliminar duplicados consecutivos
    # ------------------------------------------------------

    bloques_limpios = []


    for bloque in bloques:

        if (

            bloques_limpios

            and normalizar_texto_para_comparacion(
                bloque
            )
            ==
            normalizar_texto_para_comparacion(
                bloques_limpios[-1]
            )

        ):

            continue


        bloques_limpios.append(
            bloque
        )


    contenido = "\n\n".join(
        bloques_limpios
    )


    # ------------------------------------------------------
    # Filtrado por contenido insuficiente
    # ------------------------------------------------------

    if contenido_demasiado_corto(
        contenido
    ):

        print(
            "Descartado (contenido insuficiente)"
        )

        continue


    # ------------------------------------------------------
    # Filtrado conservador de contenido ajeno
    # ------------------------------------------------------

    if contenido_claramente_ajeno(

        titulo,

        contenido,

        url_final

    ):

        print(
            "Descartado (contenido claramente ajeno)"
        )

        continue


    # ------------------------------------------------------
    # Guardar página
    # ------------------------------------------------------

    paginas_extraidas.append({

        "url_original":
            url_original,

        "url_final":
            url_final,

        "titulo":
            titulo,

        "texto_enlace":
            enlace.get(
                "texto",
                ""
            ),

        "seccion_origen":
            enlace.get(
                "seccion_origen",
                ""
            ),

        "padre_origen":
            enlace.get(
                "padre_origen",
                ""
            ),

        "tipo_origen":
            enlace.get(
                "tipo_origen",
                ""
            ),

        "contenido":
            contenido

    })


    print(
        "Página aceptada"
    )


# ==========================================================
# RESUMEN
# ==========================================================

print()
print("=" * 80)
print("RESULTADO DE LA EXTRACCIÓN")
print("=" * 80)

print()

print(
    "URLs candidatas:",
    len(urls_candidatas)
)

print(
    "URLs finales únicas:",
    len(urls_finales_procesadas)
)

print(
    "Páginas útiles:",
    len(paginas_extraidas)
)

print()


RECOPILACIÓN DE URLS
URLs candidatas: 55

https://jpamasteres.upv.es/
Descartado (URL fuera del ámbito útil)

http://www.upv.es/rankings
Descartado (no se encontró contenido principal)

https://www.youtube.com/watch?v=MPWcjSTJZuA
Descartado (URL fuera del ámbito útil)

https://www.upv.es/otros/upv-360-es.html
Descartado (no se encontró contenido principal)

https://www.youtube.com/watch?v=uoQOKIQyc2k
Descartado (URL fuera del ámbito útil)

https://www.upv.es/entidades/adge
URL final: https://www.upv.es/entidades/adge
Página aceptada

https://www.upv.es/entidades/bvpi
URL final: https://www.upv.es/entidades/bvpi
Página aceptada

https://www.upv.es/contenidos/asistenteia
URL final: https://www.upv.es/contenidos/asistenteia
Página aceptada

https://www.upv.es/estudios/master/index-es.html
URL final: https://www.upv.es/estudios/master/index-es.html
Página aceptada

https://www.upv.es/estudios/posgrado/masteres-habilitantes-es.html
URL final: https://www.upv.es/estudios/profesiones-regulad

In [10]:
# ==========================================================
# BLOQUE 11. Generación de Markdown de páginas enlazadas
#
# Admisión Máster + metadatos YAML
# ==========================================================

import os
import re


# ----------------------------------------------------------
# Configuración
# ----------------------------------------------------------

directorio_base = os.path.join(
    ruta_programa,
    "ADMISION",
    "Master"
)


directorio_recursos = os.path.join(
    directorio_base,
    "recursos"
)


os.makedirs(
    directorio_recursos,
    exist_ok=True
)


CATEGORIA = "admision"
NIVEL = "master"


# ----------------------------------------------------------
# Limpieza de nombres de archivo
# ----------------------------------------------------------

def limpiar_nombre_recurso(nombre):

    if not nombre:
        return "recurso"

    nombre = nombre.lower()


    cambios = {

        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
        "ñ": "n"

    }


    for viejo, nuevo in cambios.items():

        nombre = nombre.replace(
            viejo,
            nuevo
        )


    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )


    nombre = nombre.strip("_")


    if not nombre:

        nombre = "recurso"


    return nombre


# ----------------------------------------------------------
# Clasificación del recurso
# ----------------------------------------------------------

def clasificar_recurso(pagina):

    texto = (

        pagina.get("titulo", "")
        + " "
        + pagina.get("texto_enlace", "")
        + " "
        + pagina.get("url_final", "")

    ).lower()


    # ------------------------------------------------------
    # Plazos y calendarios
    # ------------------------------------------------------

    if any(

        palabra in texto

        for palabra in [

            "plazo",
            "plazos",
            "calendario",
            "calendarios",
            "fecha",
            "fechas"

        ]

    ):

        return "calendario"


    # ------------------------------------------------------
    # Tasas y matrícula
    # ------------------------------------------------------

    if any(

        palabra in texto

        for palabra in [

            "precio",
            "precios",
            "tasa",
            "tasas",
            "matricula",
            "matrícula"

        ]

    ):

        return "matricula"


    # ------------------------------------------------------
    # Preguntas frecuentes
    # ------------------------------------------------------

    if any(

        palabra in texto

        for palabra in [

            "faq",
            "faqs",
            "preguntas frecuentes"

        ]

    ):

        return "faq"


    # ------------------------------------------------------
    # Becas y ayudas
    # ------------------------------------------------------

    if any(

        palabra in texto

        for palabra in [

            "ayuda",
            "ayudas",
            "beca",
            "becas"

        ]

    ):

        return "ayudas"


    # ------------------------------------------------------
    # Admisión / solicitud / preinscripción
    # ------------------------------------------------------

    if any(

        palabra in texto

        for palabra in [

            "solicitud",
            "solicitudes",
            "preinscripcion",
            "preinscripción",
            "admision",
            "admisión",
            "acceso"

        ]

    ):

        return "admision"


    # ------------------------------------------------------
    # Oferta / estudios / programas
    # ------------------------------------------------------

    if any(

        palabra in texto

        for palabra in [

            "master",
            "máster",
            "estudios",
            "oferta",
            "programa",
            "programas"

        ]

    ):

        return "programas"


    # ------------------------------------------------------
    # Normativa
    # ------------------------------------------------------

    if any(

        palabra in texto

        for palabra in [

            "normativa",
            "reglamento",
            "legislacion",
            "legislación",
            "normas"

        ]

    ):

        return "normativa"


    # ------------------------------------------------------
    # Por defecto
    # ------------------------------------------------------

    return "informacion"


# ----------------------------------------------------------
# Metadatos YAML
# ----------------------------------------------------------

def escribir_metadatos_recurso(
    f,
    pagina,
    tipo_recurso
):

    f.write(
        "---\n"
    )


    f.write(
        "fuente: UPV\n"
    )


    f.write(
        f"categoria: {CATEGORIA}\n"
    )


    f.write(
        f"nivel: {NIVEL}\n"
    )


    f.write(
        "tipo_documento: recurso\n"
    )


    f.write(
        f"tipo_recurso: {tipo_recurso}\n"
    )


    f.write(
        "seccion_origen: "
        + limpiar_nombre_recurso(
            pagina.get(
                "seccion_origen",
                ""
            )
        )
        + "\n"
    )


    # Se conserva también la modalidad / padre
    # desde la que se obtuvo el recurso.

    if pagina.get(
        "padre_origen"
    ):

        f.write(
            "padre_origen: "
            + limpiar_nombre_recurso(
                pagina["padre_origen"]
            )
            + "\n"
        )


    f.write(
        f"url: {pagina['url_final']}\n"
    )


    f.write(
        "---\n\n"
    )


# ==========================================================
# DEDUPLICACIÓN FINAL
#
# Dos URLs diferentes pueden haber terminado apuntando
# a la misma página después de las redirecciones.
# ==========================================================

paginas_unicas = []

urls_vistas = set()


for pagina in paginas_extraidas:

    url_final = pagina.get(
        "url_final"
    )


    if not url_final:
        continue


    if url_final in urls_vistas:

        continue


    urls_vistas.add(
        url_final
    )


    paginas_unicas.append(
        pagina
    )


# ==========================================================
# GENERACIÓN DE MARKDOWN
# ==========================================================

contador = 0


for pagina in paginas_unicas:

    # ------------------------------------------------------
    # Clasificación
    # ------------------------------------------------------

    tipo_recurso = clasificar_recurso(
        pagina
    )


    # ------------------------------------------------------
    # Nombre del archivo
    # ------------------------------------------------------

    nombre = limpiar_nombre_recurso(
        pagina.get(
            "titulo",
            ""
        )
    )


    archivo = os.path.join(

        directorio_recursos,

        f"{nombre}.md"

    )


    # ------------------------------------------------------
    # Evitar colisiones de nombres
    # ------------------------------------------------------

    if os.path.exists(
        archivo
    ):

        seccion = limpiar_nombre_recurso(
            pagina.get(
                "seccion_origen",
                ""
            )
        )


        nombre_base = nombre


        if seccion:

            nombre = (
                f"{nombre_base}_{seccion}"
            )


        archivo = os.path.join(

            directorio_recursos,

            f"{nombre}.md"

        )


    # Si sigue existiendo, añadir contador.

    contador_colision = 2


    while os.path.exists(
        archivo
    ):

        archivo = os.path.join(

            directorio_recursos,

            f"{nombre}_{contador_colision}.md"

        )


        contador_colision += 1


    # ------------------------------------------------------
    # Escritura del Markdown
    # ------------------------------------------------------

    with open(

        archivo,

        "w",

        encoding="utf-8"

    ) as f:


        # ----------------------------------------------
        # YAML
        # ----------------------------------------------

        escribir_metadatos_recurso(

            f,

            pagina,

            tipo_recurso

        )


        # ----------------------------------------------
        # Título
        # ----------------------------------------------

        f.write(
            f"# {pagina['titulo']}\n\n"
        )


        # ----------------------------------------------
        # Contexto
        # ----------------------------------------------

        f.write(
            "Recurso relacionado con el proceso de "
            "admisión a estudios oficiales de máster "
            "de la Universitat Politècnica de València.\n\n"
        )


        # ----------------------------------------------
        # Procedencia
        # ----------------------------------------------

        if pagina.get(
            "seccion_origen"
        ):

            f.write(
                "Sección de origen: "
                + pagina["seccion_origen"]
                + "\n\n"
            )


        # ----------------------------------------------
        # Contenido
        # ----------------------------------------------

        f.write(
            pagina.get(
                "contenido",
                ""
            )
            +
            "\n\n"
        )


        # ----------------------------------------------
        # Fuente oficial
        # ----------------------------------------------

        f.write(
            "Fuente oficial: "
            + pagina["url_final"]
            + "\n"
        )


    contador += 1


# ==========================================================
# RESULTADO
# ==========================================================

print()

print("=" * 70)

print(
    "Markdown de recursos generado correctamente."
)

print()

print(
    "Páginas extraídas:",
    len(paginas_extraidas)
)

print(
    "Páginas únicas:",
    len(paginas_unicas)
)

print(
    "Archivos creados:",
    contador
)

print()

print(
    "Ruta:",
    directorio_recursos
)

print("=" * 70)


Markdown de recursos generado correctamente.

Páginas extraídas: 9
Páginas únicas: 9
Archivos creados: 9

Ruta: /content/drive/MyDrive/TFG Teleco/ADMISION/Master/recursos
